# Lab 02: Vector Search & Retrieval

## Overview

In this lab you build a production-ready **Vector Search index** on top of the
`arxiv_chunks` Delta table created in Lab 01.  You will:

1. Provision a **Vector Search Endpoint** (the compute that serves queries).
2. Create a **Delta Sync index** with **Managed Embeddings** using
   `databricks-bge-large-en` — Databricks handles embedding generation and
   incremental sync automatically.
3. Query the index using **semantic**, **keyword (BM25)**, and **hybrid** search.
4. Apply a **reranker** model to boost result quality.

**Exam Domain:** Data Preparation — *14 % of exam*

Relevant skills: Delta Sync indexes, managed embeddings, query types,
reranker usage, and embedding-model trade-offs.

## Architecture Diagram

![Architecture Diagram](../assets/diagrams/lab-02-vector-search-retrieval.png)

In [ ]:
# Prerequisites — run once per session
%pip install databricks-sdk databricks-vectorsearch --quiet
dbutils.library.restartPython()

In [ ]:
# Configuration
CATALOG = "genai_lab_guide"
SCHEMA = "default"
SOURCE_TABLE = f"{CATALOG}.{SCHEMA}.arxiv_chunks"
VS_ENDPOINT = "genai_lab_guide_vs_endpoint"
VS_INDEX = f"{CATALOG}.{SCHEMA}.arxiv_chunks_index"

print(f"Source table: {spark.table(SOURCE_TABLE).count()} chunks")

## A. Create a Vector Search Endpoint

A **Vector Search Endpoint** is the compute resource that hosts and serves one or
more Vector Search indexes.  Endpoints are shared resources — a single endpoint can
serve many indexes — and they autoscale automatically.

Key points:
- You must create an endpoint before you can create any index.
- Endpoint names are workspace-scoped.
- The same endpoint can serve indexes backed by different embedding models.

In [ ]:
from databricks.vector_search.client import VectorSearchClient
import time

vsc = VectorSearchClient()

# Create or reuse existing endpoint (trial workspaces allow only 1)
try:
    vsc.create_endpoint(name=VS_ENDPOINT, endpoint_type="STANDARD")
    print(f"Creating endpoint '{VS_ENDPOINT}'...")
except Exception as e:
    if "already exists" in str(e).lower() or "quota" in str(e).lower():
        print(f"Endpoint '{VS_ENDPOINT}' already exists, reusing.")
    else:
        raise

# Wait for endpoint to be online
for i in range(30):
    try:
        ep = vsc.get_endpoint(VS_ENDPOINT)
        status = ep.get("endpoint_status", {}).get("state", "UNKNOWN")
        print(f"  Endpoint status: {status}")
        if status == "ONLINE":
            break
    except:
        pass
    time.sleep(10)
else:
    print("WARNING: Endpoint may not be ready yet. Proceed and retry if needed.")

print(f"Endpoint '{VS_ENDPOINT}' is ready.")

## B. Create a Delta Sync Vector Search Index

A **Delta Sync index** stays in sync with the source Delta table automatically.
By choosing **Managed Embeddings** you delegate embedding generation to Databricks
using `databricks-bge-large-en` — you do not need to call an embedding model
yourself.

Prerequisites (already satisfied from Lab 01):
1. Source table has a **unique primary key** column (`chunk_id`).
2. Source table has **Change Data Feed** (`delta.enableChangeDataFeed = true`) enabled.

`pipeline_type="TRIGGERED"` means the index refreshes on demand (cost-efficient
for a lab).  In production you would use `"CONTINUOUS"` for near-real-time updates.

In [ ]:
# Create Delta Sync index with managed embeddings
try:
    index = vsc.create_delta_sync_index(
        endpoint_name=VS_ENDPOINT,
        index_name=VS_INDEX,
        source_table_name=SOURCE_TABLE,
        pipeline_type="TRIGGERED",
        primary_key="chunk_id",
        embedding_source_column="chunk_text",
        embedding_model_endpoint_name="databricks-bge-large-en"
    )
    print(f"Creating index '{VS_INDEX}'...")
except Exception as e:
    if "already exists" in str(e).lower():
        print(f"Index '{VS_INDEX}' already exists, reusing.")
    else:
        raise

# Wait for index to be ready
print("Waiting for index to sync (may take several minutes)...")
for i in range(60):
    try:
        idx = vsc.get_index(endpoint_name=VS_ENDPOINT, index_name=VS_INDEX)
        status = idx.describe().get("status", {}).get("ready", False)
        detailed = idx.describe().get("status", {})
        print(f"  Index status: ready={status} | {detailed.get('message', '')}")
        if status:
            break
    except Exception as e:
        print(f"  Checking: {e}")
    time.sleep(15)
else:
    print("WARNING: Index may not be fully synced yet.")

print(f"Index '{VS_INDEX}' is ready.")

## C. Semantic Search

**Semantic search** finds documents whose *meaning* is similar to the query,
even when no exact words match.  Under the hood Databricks:

1. Embeds the query text using the same model as the index (`databricks-bge-large-en`).
2. Computes cosine similarity between the query embedding and every stored embedding.
3. Returns the top-*k* chunks sorted by similarity score.

Use semantic search when the user query is expressed in natural language and you
want conceptual matches rather than exact keyword hits.

In [ ]:
index = vsc.get_index(VS_ENDPOINT, VS_INDEX)

# Semantic search
results = index.similarity_search(
    query_text="How does the attention mechanism work in transformers?",
    columns=["chunk_id", "path", "chunk_text"],
    num_results=5
)

# Display results
print("=== Semantic Search Results ===")
result_data = results.get("result", {}).get("data_array", [])
if result_data:
    for doc in result_data:
        score = doc[-1] if len(doc) > 3 else "N/A"
        source = doc[1].split("/")[-1] if len(doc) > 1 else "unknown"
        text = doc[2][:200] if len(doc) > 2 else ""
        print(f"Score: {score} | Source: {source}")
        print(f"  {text}...")
        print()
else:
    print("No results returned. Index may still be syncing.")

## D. Keyword Search

**Keyword search** uses BM25 (Best Matching 25), a classic term-frequency ranking
algorithm.  It scores documents based on how often query terms appear relative to
the document length and the overall corpus frequency.

BM25 excels when:
- The query contains specific technical terms, acronyms, or proper nouns.
- Exact terminology matters (e.g., "BERT", "LoRA", "GPT-4").
- The concept is not well represented in the embedding space.

BM25 struggles with synonyms and paraphrasing — a query for "car" will not match
a document that only uses "automobile".

In [ ]:
results_kw = index.similarity_search(
    query_text="BERT masked language model pre-training",
    columns=["chunk_id", "path", "chunk_text"],
    num_results=5,
    query_type="FULL_TEXT",
)

print("=== Keyword (BM25) Search Results ===")
result_data = results_kw.get("result", {}).get("data_array", [])
if result_data:
    for doc in result_data:
        score = doc[-1] if len(doc) > 3 else "N/A"
        source = doc[1].split("/")[-1] if len(doc) > 1 else "unknown"
        text = doc[2][:200] if len(doc) > 2 else ""
        print(f"Score: {score} | Source: {source}")
        print(f"  {text}...")
        print()
else:
    print("No results returned. Index may still be syncing.")

## E. Hybrid Search

**Hybrid search** combines semantic (embedding) similarity with keyword (BM25)
matching using Reciprocal Rank Fusion (RRF).  RRF merges ranked lists from both
retrievers: a chunk that appears high in *both* lists gets a strong combined score.

When to use hybrid:
- General-purpose RAG pipelines where query types are unpredictable.
- Queries that mix natural language with technical terms.
- When you want the precision of keyword search and the recall of semantic search.

Hybrid is the recommended default for most production RAG applications.

In [ ]:
results_hybrid = index.similarity_search(
    query_text="Low-rank adaptation fine-tuning efficiency",
    columns=["chunk_id", "path", "chunk_text"],
    num_results=5,
    query_type="HYBRID",
)

print("=== Hybrid Search Results ===")
result_data = results_hybrid.get("result", {}).get("data_array", [])
if result_data:
    for doc in result_data:
        score = doc[-1] if len(doc) > 3 else "N/A"
        source = doc[1].split("/")[-1] if len(doc) > 1 else "unknown"
        text = doc[2][:200] if len(doc) > 2 else ""
        print(f"Score: {score} | Source: {source}")
        print(f"  {text}...")
        print()
else:
    print("No results returned. Index may still be syncing.")

## F. Comparing Embedding Models

Databricks hosts several BGE models out of the box.  Choosing the right one
involves a cost-vs-quality trade-off:

| Model | Size | Dimension | Throughput | Best for |
|-------|------|-----------|------------|----------|
| `databricks-bge-large-en` | 0.44 GB | 1 024 | Medium | Higher accuracy, complex queries |
| `databricks-bge-small-en` | 0.13 GB | 384 | High | Cost-sensitive, high-volume pipelines |

**Key trade-offs:**

- **Higher dimension** → richer representation → better semantic recall, but more
  storage and slower ANN search.
- **Larger model** → higher embedding quality, but more GPU memory and higher
  inference cost per token.
- Once an index is created with a given model, changing the model requires
  **rebuilding the index from scratch** — choose wisely upfront.
- For the exam: `bge-large` = better quality; `bge-small` = lower cost.

## G. Adding a Reranker

A **reranker** is a cross-encoder model that re-scores an initial set of
retrieved candidates.  Unlike the bi-encoder used for retrieval (which encodes
query and document independently), a reranker processes both together — giving it
much better relevance judgement.

Reranker workflow:

1. Retrieve *N* candidate chunks via hybrid search (e.g., top 20).
2. Pass each (query, chunk) pair through the reranker.
3. Re-sort by reranker score and return the top *k* (e.g., top 5).

Important nuances:
- A reranker **does not add new documents** — it only reorders what retrieval already found.
- It adds latency and cost, so use it when precision matters more than speed.
- Databricks provides `databricks-reranker-v1` as a managed endpoint.

In [ ]:
# Search with reranker (if available)
try:
    results_reranked = index.similarity_search(
        query_text="How does chain-of-thought prompting improve reasoning?",
        columns=["chunk_id", "path", "chunk_text"],
        num_results=5,
        query_type="HYBRID",
        reranker={"reranker_model_endpoint_name": "databricks-reranker-v1"}
    )
    print("Results WITH reranker:")
    for doc in results_reranked.get("result", {}).get("data_array", []):
        score = doc[-1] if len(doc) > 3 else "N/A"
        source = doc[1].split("/")[-1] if len(doc) > 1 else "unknown"
        print(f"  Score: {score} | {source}")
        print(f"    {doc[2][:150]}...")
        print()
except Exception as e:
    print(f"Reranker not available: {e}")
    print("Running without reranker instead...")
    results_no_rerank = index.similarity_search(
        query_text="How does chain-of-thought prompting improve reasoning?",
        columns=["chunk_id", "path", "chunk_text"],
        num_results=5,
        query_type="HYBRID"
    )
    for doc in results_no_rerank.get("result", {}).get("data_array", []):
        score = doc[-1] if len(doc) > 3 else "N/A"
        source = doc[1].split("/")[-1] if len(doc) > 1 else "unknown"
        print(f"  Score: {score} | {source}")

## Cleanup — Stop the Billing Clock

The Vector Search endpoint costs ~$0.50-1.00/hr while running. If you're done for the day, delete it. You can recreate it in Lab 02 when you resume.

In [ ]:
# Uncomment to delete the Vector Search endpoint (saves ~$0.50-1.00/hr)
# vsc.delete_endpoint(VS_ENDPOINT)
# print(f"Deleted endpoint: {VS_ENDPOINT}")

## Key Takeaways

1. **Endpoint first, index second.** A Vector Search Endpoint must exist before you
   can create any index.  One endpoint serves many indexes.

2. **Delta Sync prerequisites.** The source Delta table must have a unique primary
   key and Change Data Feed enabled — both were set up in Lab 01.

3. **Managed Embeddings = zero embedding code.** Databricks auto-generates and
   stores embeddings when `embedding_source_column` and
   `embedding_model_endpoint_name` are specified.

4. **Semantic vs. Keyword vs. Hybrid.**  Use semantic for natural-language queries,
   keyword for exact terms, and hybrid (recommended default) for mixed workloads.
   Hybrid uses Reciprocal Rank Fusion under the hood.

5. **Rerankers improve precision, not recall.** They reorder existing candidates —
   they do not surface new documents.  Apply them when the top-5 quality matters
   more than raw throughput.

6. **Embedding model trade-off.** `bge-large` (1 024 dim) delivers better quality;
   `bge-small` (384 dim) is faster and cheaper.  Changing the model requires
   rebuilding the index.

---

question patterns, and a cost breakdown.

## Key Concepts

| Concept | Definition |
|---------|-----------|
| VS Endpoint | Compute resource that hosts and serves Vector Search indexes; must be created before any index |
| Delta Sync Index | Index type that auto-syncs with a source Delta table using CDF; supports managed and self-managed embeddings |
| Managed Embeddings | Databricks handles embedding generation using a specified model endpoint — no user code needed |
| Semantic Search | ANN-based retrieval using cosine similarity between query and document embeddings |
| Keyword Search (BM25) | Term-frequency ranking algorithm; effective for exact technical terms and acronyms |
| Hybrid Search | Combines semantic + keyword results via Reciprocal Rank Fusion (RRF); recommended default for RAG |
| Reranker | Cross-encoder model that re-scores a candidate set for higher precision; does not expand the candidate pool |

## Exam Preparation

### Common Exam Question Patterns

1. **"What are the prerequisites for creating a Delta Sync Vector Search index?"**
   → Unique primary key column + Change Data Feed (`delta.enableChangeDataFeed = true`)
   on the source Delta table.  Both must be set before calling `create_delta_sync_index()`.

2. **"When should you use hybrid search instead of semantic or keyword search?"**
   → When queries are unpredictable or mix natural language with specific technical
   terms.  Hybrid (RRF) is the recommended default for production RAG pipelines.

3. **"A reranker improves result quality, but which documents does it consider?"**
   → Only the candidates already returned by retrieval.  A reranker reorders —
   it never adds new documents from outside the initial result set.

4. **"Which embedding model should you choose for a cost-sensitive, high-volume pipeline?"**
   → `databricks-bge-small-en` (384 dim, 0.13 GB) — lower compute and storage cost
   at the expense of some retrieval quality vs. `bge-large`.

5. **"A user queries for 'GPU memory optimization techniques' but only gets results
   mentioning 'VRAM'.  Which search type would help most?"**
   → Semantic search — it captures conceptual similarity and will match "VRAM"
   as semantically related to "GPU memory", unlike keyword (BM25) which requires
   exact term overlap.

## Cost Breakdown

| Resource | Usage | Est. Cost |
|----------|-------|-----------|
| Serverless Compute | ~20 min notebook execution | ~$0.50 |
| Managed Embeddings | Embedding all chunks on index creation | ~$0.50 |
| VS Endpoint | Running during lab (~10 min active) | ~$1.00 |
| VS Queries | ~15 similarity_search calls | ~$0.10 |
| Reranker inference | ~5 reranked calls × 20 candidates | ~$0.10 |
| **Total** | | **~$2-3** |

**Estimated time:** ~30 min | **Estimated cost:** ~$2-3 (endpoint compute + managed embeddings + VS queries)